# One function, any input

Every screamer function is built once, then called on whatever you have — a NumPy
array, several series at once, or values arriving one at a time from a live feed.
The numbers come out the same either way, so you can develop and test on stored
data and then run the *exact same code* in production. This is screamer's
**polymorphic API**; the [reference](../polymorphic_api.md) gives the exact rule
for every input type.

Below, one function — `RollingMean` — does all of it.

In [ ]:
import numpy as np
from screamer import RollingMean

rng = np.random.default_rng(0)
mean5 = RollingMean(5)      # the function: a 5-sample rolling mean

## A whole array

Pass an array, get an array back. The first four values are `NaN` because the
5-sample window is not full yet. (A Python list works the same way.)

In [ ]:
data = rng.standard_normal(10)
batch = mean5(data)
print(batch.round(3))

## Several series at once

The first axis is time; every other axis is an independent series. A `(10, 3)`
array is three series processed side by side, with no loop of your own.

In [ ]:
series = rng.standard_normal((10, 3))
print(mean5(series).shape)      # (10, 3): three independent rolling means

## One value at a time

The live case: values arrive one by one and you feed each into the same function
as it comes. Passing a generator instead returns a lazy iterator you can advance
inside an event loop — see the [streaming example](06-streaming-live-events.ipynb).

In [ ]:
streamed = [mean5(x) for x in data]   # imagine each x arriving from a socket or clock

## Stored and live give the same result

The array computed in one pass and the values fed one at a time match exactly.

In [ ]:
np.testing.assert_allclose(batch, streamed, equal_nan=True)
print("stored and live results match")